In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.functional as F


In [53]:
f = open(r'C:\Users\devan\nlp\practice\train.hi',encoding='utf8')
w1 = f.readlines()
print(len(w1))
print(w1[:10])
g = open(r'C:\Users\devan\nlp\practice\train.en',encoding='utf8')
w2 = g.readlines()
print(len(w2))
print(w2[:10])

84557
['और उनके Sigil क्या है?\n', 'मैं मरना नहीं चाहता.\n', 'यह मुझे लगता है कि एक ही देश है.\n', 'फिर ये नन्हें बच्चों की तरह रोएँगे।\n', 'नहीं, मुझे पावर की जरुरत है !\n', 'मैं उसे नहीं खा जाएगा.\n', 'आप चार्ल्सटन करने के लिए मुझे जाना होगा.\n', '- नहीं, वह मेरे पिता नहीं है.\n', 'मैं रविवार को उसे हम बाकी बताया.\n', 'तुम्हें कम से कम मुझे तो बताना चाहिए था,ना?\n']
84557
['And what is their Sigil?\n', 'I do not want to die.\n', "It's the same country I think.\n", "Then they'll be crying like babies.\n", '- No, I need power up!\n', 'I will not eat him.\n', 'You gotta get me to Charleston.\n', "- NO, HE'S NOT MY DAD.\n", 'I told her we rest on Sundays.\n', "You could've at least informed me, right?\n"]


In [54]:
num = 70000
ip =[]
op = []
count=0
for line in open(r'train.en', encoding="utf-8"):
    count += 1

    if count > num:
        break

    ip_w = line.rstrip().strip("\n").strip('-') #strip the sentence of '\n' and '-' 
    ip.append(ip_w) #store all input sentences in the input sentences list

count = 0

for line in open(r'train.hi', encoding="utf-8"):
    count += 1

    if count > num:
        break
    op_w= line.rstrip().strip("\n").strip('-') 
    from indicnlp.tokenize import indic_tokenize  
    line = indic_tokenize.trivial_tokenize(op_w) #we tokenize the hindi sentences 

    op.append(['<sos>'] + line + ['<eos>']) 

In [55]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [56]:
class encoder(nn.Module):
    def __init__(self,ip_dim,emb_dim,hid_dim,dropout,layers): 
        super().__init__()
        self.embedding=nn.Embedding(ip_dim,emb_dim)
        self.rnn=nn.GRU(emb_dim,hid_dim,layers,batch_first=True)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
            embeded=self.dropout(self.embedding(x))
            output,hidden=self.rnn(embeded)
            return output,hidden

In [57]:
class attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [layers, batch, hid_dim]
        # encoder_outputs: [batch, src_len, hid_dim]
        
        if hidden.dim() == 3:
            hidden = hidden[-1] 
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)  # [batch, src_len, hid_dim]
        # Calculate Alignment Scores (Energy)
        combined = torch.cat((hidden, encoder_outputs), dim=2)
        energy = torch.tanh(self.attn(combined))
        
        # Calculate Attention Weights
        attention_weights = self.v(energy).squeeze(2) 
        
        # Use torch.softmax to avoid 'F' attribute errors
        return torch.softmax(attention_weights, dim=1)

In [58]:
class decoder(nn.Module):
    def __init__(self, op_dim, emb_dim, hid_dim, dropout, layers, attention):
        super().__init__()
        self.attention = attention
        self.op_dim = op_dim
        self.embedding = nn.Embedding(op_dim, emb_dim)
        self.ffn = nn.Linear(hid_dim * 2 + emb_dim, op_dim) 
        self.rnn = nn.GRU(hid_dim + emb_dim, hid_dim, layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
    def forward(self, input, hidden, emd_op):
    
      if hidden.dim() == 3:
        hidden_for_attn = hidden[-1] # Take the last layer
      else:
        hidden_for_attn = hidden
      ip = input.unsqueeze(1) # [batch_size, 1]
      embeded = self.dropout(self.embedding(ip)) # [batch_size, 1, emb_dim]
      # 2. Get attention context
      a = self.attention(hidden_for_attn, emd_op)
      a = a.unsqueeze(1) # [batch_size, 1, src_len]
      weighted = torch.bmm(a, emd_op) # [batch_size, 1, hid_dim]
      # 3. Prepare GRU input
      rnn_ip = torch.cat((embeded, weighted), dim=2) # [batch_size, 1, emb_dim + hid_dim]
      # 4. GRU step
      # GRU strictly wants [layers, batch_size, hid_dim]
      if hidden.dim() == 2:
        hidden_for_gru = hidden.unsqueeze(0)
      else:
        hidden_for_gru = hidden

      output, hidden = self.rnn(rnn_ip, hidden_for_gru) 
    
      # 5. Prepare for Linear Layer (FFN)
      output = output.squeeze(1)     
      weighted = weighted.squeeze(1) 
      embeded = embeded.squeeze(1)   
    
      prediction = self.ffn(torch.cat((output, weighted, embeded), dim=1))
    
      # 6. Return hidden state
      # Keeping it as [layers, batch, hid_dim] is usually safest for GRUs
      return prediction, hidden

In [59]:
torch.manual_seed(42)
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, tfr=0.5):#teacher_forcing_ratio
        bs = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.op_dim
        outputs = torch.zeros(bs, trg_len, trg_vocab_size).to(self.device)
        enc_op, hidden = self.encoder(src)
        
        # first input to decoder is <sos>
        input = trg[:, 0]
        
        for t in range(1, trg_len):
            output, hidden = self.decoder(input, hidden, enc_op)
            outputs[:, t, :] = output
            top1 = output.argmax(1) 
            input = trg[:, t] if torch.rand(1) < tfr else top1
            
        return outputs

In [60]:
from collections import Counter
def build_vocab(sentences, min_freq=2):
    tokens = []
    for sent in sentences:
        tokens.extend(sent if isinstance(sent, list) else sent.split())
    counter = Counter(tokens)
    # Define special tokens: pad (0), start (1), end (2), unknown (3)
    vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab
en_vocab = build_vocab(ip)
hi_vocab = build_vocab(op)

PyTorch needs a DataLoader to feed data in batches. Since sentences have different lengths, we use pad_sequence to make them uniform within a batch.

In [61]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, trg_sentences, src_vocab, trg_vocab):
        self.src_data = src_sentences
        self.trg_data = trg_sentences
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        src = [self.src_vocab.get(w, 3) for w in self.src_data[idx].split()]
        trg = [self.trg_vocab.get(w, 3) for w in self.trg_data[idx]]
        return torch.tensor(src), torch.tensor(trg)

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=0, batch_first=True)
    return src_batch, trg_batch

dataset = TranslationDataset(ip, op, en_vocab, hi_vocab)
loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

In [62]:
Ip_dim = len(en_vocab)
output_dim = len(hi_vocab)
Enc_emb_dim = 256
DEC_EMB_DIM = 256
HID_DIM = 256
N_LAYERS = 1
enc = encoder(Ip_dim, Enc_emb_dim, HID_DIM, 0.5, N_LAYERS).to(device)
attn = attention(HID_DIM).to(device)
dec = decoder(output_dim, DEC_EMB_DIM, HID_DIM, 0.5, N_LAYERS, attn).to(device)
model = Seq2Seq(enc, dec, device).to(device)
import time
optimizer = torch.optim.Adam(model.parameters(),lr=5e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    time_start=time.time()
    model.train()
    epoch_loss = 0
    for src, trg in loader:
        # Change this line (Line 20 in your traceback)
        src, trg = src.to(device).long(), trg.to(device).long() 

        # Now the model call will work
        optimizer.zero_grad()
        output = model(src, trg)  
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        time_end=time.time()
    
    print(f'Epoch: {epoch+1} | Loss: {epoch_loss/len(loader):.4f} | Time: {time_end-time_start:.2f}s')

Epoch: 1 | Loss: 2.1304 | Time: 181.70s
Epoch: 2 | Loss: 1.8419 | Time: 180.17s
Epoch: 3 | Loss: 1.7221 | Time: 209.57s
Epoch: 4 | Loss: 1.6302 | Time: 178.66s
Epoch: 5 | Loss: 1.5520 | Time: 179.73s
Epoch: 6 | Loss: 1.5038 | Time: 177.88s
Epoch: 7 | Loss: 1.4511 | Time: 179.77s
Epoch: 8 | Loss: 1.4019 | Time: 178.94s
Epoch: 9 | Loss: 1.3780 | Time: 179.27s
Epoch: 10 | Loss: 1.3419 | Time: 179.67s


In [78]:
english = input("Enter the english sentence: ")
print(f' Translating: {english}')
model.eval()

with torch.no_grad():
    # 1. Preprocessing (Force Long)
    src_indices = [en_vocab.get(w.lower(), 3) for w in english.split()]
    src_tensor = torch.tensor(src_indices).long().unsqueeze(0).to(device)
    
    # 2. Encoder
    enc_op, hidden = model.encoder(src_tensor)
    
    # Start with <sos>
    trg_indices = [hi_vocab['<sos>']]
    
    # 3. Decoder Loop
    for _ in range(50):
        trg_tensor = torch.tensor([trg_indices[-1]]).long().to(device)
        
        output, hidden = model.decoder(trg_tensor, hidden, enc_op)
        pred_token = output.argmax(1).item()
        
        trg_indices.append(pred_token)
        if pred_token == hi_vocab.get('<eos>', None):
            break
    
    # 4. Post-processing
    inv_hi_vocab = {v: k for k, v in hi_vocab.items()}
    translated_tokens = [inv_hi_vocab.get(idx, '<unk>') for idx in trg_indices]
    
    # Clean up special tokens
    stop_signals = ['<eos>', '.', '।', '!', '?']
    final_output = []
    for t in translated_tokens:
        if t in stop_signals:
            break  # Immediately stop when the model finishes the thought
        if t not in ['<sos>', '<pad>', '<unk>']:
            final_output.append(t)
    print("Translated Hindi Sentence:", ' '.join(final_output))

 Translating: wait for me
Translated Hindi Sentence: और मेरे लिए इंतजार करेंगे


In [79]:
torch.save(model.state_dict(), 'seq2seq_model.pt')
